# 17 · 第四个工具：找相似的病人（病例检索）

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → [17 找相似病人 ✅] → 18 疾病介绍（RAG） → 19 基因工具 → 20 大考
```

> **这一章的代码在合并请求 #19 里。** 不用模型的评测免费；调模型的只有一次 20 道题的试跑（约 ¥0.5），记在 MLflow 里。下面每一格都只读记录、在本机计算，**运行这本笔记不花钱**。

## 背景：排名漏掉的病，别的病人身上有

第 16 章的错题分析里，② 类有 39 道：正确答案在资料库里，却排得太靠后，agent 根本没看到。其中 33 道，Phenopacket Store 里**别的论文**报过得同一种病的病人。

我们的排名工具是拿病人去和**病**比：每个病的资料，是把很多篇报告合在一起的"平均画像"。可一个具体的病人，常常和平均画像不太像，却和另一个具体的病人很像。

医生也这么做：遇到怪病，去文献里找"和这个病人很像的病例"。日本的 PubCaseFinder 就是专门干这个的。这叫**病例检索**（case-based retrieval），是 RAG 的一种：先检索，再交给模型判断。只不过检索的不是文字，是病人。

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 做工具：拿病人和六千个已发表的病人比 | hpo3，和排名工具同一套相似度 | 找到排名漏掉的病 |
| 2 | 不用模型，先考工具自己 | 新基线 `similar-patients` | 工具本身值多少 |
| 3 | 逐题比：和按病排名比 | `eval compare` | 是真进步还是碰巧 |
| 4 | 两种方法合起来 | 前 10 名取并集 | 它们找到的病不一样 |
| 5 | 考试公平 | 遮住本篇论文；病例库只用 design | 不偷看答案、不偷看留出集 |
| 6 | 接进 agent 和 MCP，能拿掉 | `--without` | 为第 20 章的消融实验做准备 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 做工具：拿病人和病人比

### 在干什么
新工具 `find_similar_patients`：给一组症状，在已发表的病人里找最像的几个，连同他们确诊的病一起返回。

### 背景
举一道真实的题。一个病人有 7 个免疫方面的异常（抗体偏低、B 细胞少……），正确答案是"常见变异型免疫缺陷 13 型"。按病排名，这个病排在 50 名以外。可是拿这个病人去和已发表的病人比，会看到什么？

### 用了什么技术，为什么
- **相似度和排名工具一模一样**（hpo3，Resnik，funSimAvg），只是比的对象从"病的平均画像"换成"一个个病人"
- **病例库**：Phenopacket Store 里 design 那一半的 6,014 个病人，每人都有确诊的病和出处论文
- **每个找到的病人附上"共同症状"**：用 HPO 的上下级关系判断，和第 12 章核对用的是同一套
- 和别的工具一样：输入输出有固定格式、有测试、每次调用都记进追踪

### 公司里什么情况怎么做
- **检索不一定是向量检索**：数据本身有结构（这里是 HPO 编号），就直接用结构算相似度，比把它转成文字再做向量更准，也不用多维护一个向量库
- **返回的每一条都带出处**：哪篇论文报的，模型和医生都能回头查

### 执行

In [ ]:
import mlflow
from raredx.config import settings
from raredx.data import cases as case_store, sources
from raredx.evals.harness import Harness, candidates_of
from raredx.knowledge import hpo as hpo_knowledge, masking
from raredx.tools.find_similar_patients import FindSimilarPatients
from raredx.tools.rank_diseases import RankDiseases

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
cases = case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))
by_id = {c.id: c for c in cases}
bank = [c for c in cases if c.split == "design"]
find_similar = FindSimilarPatients(hpo, bank, f"design split: {len(bank)} patients")
rank = RankDiseases(hpo)
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("playground")

case = by_id["PMID_27939403_E_II_1"]
print("病人的症状：", "、".join(case.hpo_labels))
print("正确答案：", case.disease_id, case.disease_label, "\n")
with masking.masked([case.pmid]):                 # 和考试时一样，遮住这道题自己的论文
    ranked = [m.disease_id for m in rank(case.hpo_ids, top_k=50).matches]
    similar = find_similar(case.hpo_ids, top_k=5)
print("按病排名：", f"第 {ranked.index(case.disease_id) + 1} 名" if case.disease_id in ranked else "前 50 名里没有", "\n")
print(f"最像的 5 个已发表病人（在 {similar.searched} 人里找）：")
for p in similar.patients:
    print(f"  {p.disease_id}  {p.disease_name:<45} 相似度 {p.similarity:.2f}  出处 {p.source}")
    print(f"      共同症状：{'、'.join(p.shared)}")

### 结果怎么看
- **按病排名：前 50 名里都没有它**
- **最像的 5 个病人，全是这个病**，来自另外三篇论文。共同症状一目了然：IgA、IgG 偏低，B 细胞少，CD4/CD8 比例倒置
- 在 6,005 人里找，不是 6,014：这道题自己那篇论文的 9 个病人被遮住了（第 5 步）

### 面试怎么说
> "The disease ranking compares a patient with each disease's pooled profile; this tool compares the patient with individual published patients. Here the right disease was outside the ranking's top 50, and all five nearest patients — from three other papers — had it."

---

## 第 2 步 · 不用模型，先考工具自己

### 在干什么
做一个不用模型的基线 `similar-patients`：找最像的病人，按他们的病排出前 10 个（每个病只算一次，按它最像的那个病人排）。在 dev-200 上考。

### 背景
工具接进 agent 之前，先单独考它：它自己能找对多少？这样以后 agent 的分数变了，才知道多少是工具的功劳。第 10 章的排名工具也是这样，先有了基线 B0（19.0%）。

### 用了什么技术，为什么
和别的系统一样，用 `raredx eval run --system similar-patients` 考，记进 MLflow。不调模型，不花钱，一分钟左右。

### 公司里什么情况怎么做
- **检索单独评测**：做 RAG 的公司都会单独量"检索找得准不准"（recall@k），再看整个系统。检索没找到的，模型再聪明也答不对

### 执行

In [ ]:
def latest(name, split="dev-200"):
    return next(r for r in harness.runs(split) if r["name"] == name and not r["code"].endswith("-dirty"))

knn_run, rank_run = latest("similar-patients"), latest("phenotype-rank")
for label, record in (("按病排名（第 10 章）", rank_run), ("找相似病人", knn_run)):
    m = harness.client.get_run(record["run_id"]).data.metrics
    print(f"{label:<10} 第 1 名 {m['disease_top1/mean']:.1%}   前 3 名 {m['disease_top3/mean']:.1%}   前 10 名 {m['disease_top10/mean']:.1%}")

### 结果怎么看

| 不用模型 | 第 1 名 | 前 10 名 |
|---|---|---|
| 按病排名（第 10 章） | 19.0% | 49.0% |
| **找相似病人** | **29.0%** | 46.5% |

- **第 1 名高了 10 个百分点**：最像的那个病人，常常就是同一种病
- 前 10 名差不多

### 面试怎么说
> "Before the agent used it, the tool was evaluated alone: ranking diseases by their nearest published patients gets 29.0% top-1 on dev-200, against 19.0% for the disease ranking — no model involved."

---

## 第 3 步 · 逐题比

### 在干什么
同样 200 道题，逐题比"找相似病人"和"按病排名"。

### 背景
第 14 章讲过：两个系统在同一批题上比，要逐题看谁对谁错，再算这个差距会不会是碰巧（符号检验）。

### 执行

In [ ]:
for k in (1, 10):
    run(f"uv run raredx eval compare {knn_run['run_id']} {rank_run['run_id']} --k {k}")

### 结果怎么看
- **第 1 名**：只有"找相似病人"对的 35 道，只有"按病排名"对的 15 道，`p = 0.007`。**这个差距不是碰巧**
- **前 10 名**：21 比 26，打平。但注意：各自都有二十多道**对方没找到**的题，说明两种方法找到的病很不一样（下一步）

### 面试怎么说
> "Paired on the same 200 cases, nearest-patient ranking wins at first place 35 cases to 15, p = 0.007; at top 10 it ties, 21 to 26 — but each finds about two dozen diseases the other misses."

---

## 第 4 步 · 两种方法合起来

### 在干什么
算一下：两种方法的前 10 名合在一起，正确答案出现在里面的比例。

### 背景
agent 拿到两个工具，就能同时看两份名单。所以要问的是：**两份名单合起来**能覆盖多少？这是 agent 能选对的上限之一：名单里没有的，它只能靠自己想到。

### 执行

In [ ]:
def top10(record):
    answers = {}
    for t in harness.traces(record["run_id"]):
        truth = next(a.value for a in t.info.assessments if a.name == "disease_id")
        answers[t.data.spans[0].inputs["question"]["case_id"]] = (truth, [c.disease_id for c in candidates_of(t.data.spans[0].outputs)][:10])
    return answers

by_rank, by_knn = top10(rank_run), top10(knn_run)
either = sum(truth in by_rank[cid][1] or truth in by_knn[cid][1] for cid, (truth, _) in by_rank.items())
print(f"前 10 名合起来：{either / 2:.1f}%（按病排名 49.0%，找相似病人 46.5%）")

### 结果怎么看
- **合起来 59.5%**，比任何一个单独用都高 10 个百分点以上
- 两种方法看问题的角度不同：一个比"病"，一个比"人"。所以要把两个都交给 agent，而不是二选一

### 面试怎么说
> "The two retrievers are complementary: either one's top 10 holds the right disease about half the time, the two together 59.5%. So the agent gets both."

---

## 第 5 步 · 考试公平

### 在干什么
确认两件事：这个工具不会找到"这道题本人"，也不会偷看留出集。

### 背景
病例库里存的是已发表的病人，**考题本身也在里面**。不处理的话，工具会找到这道题自己（相似度满分），那就是直接看答案。同一篇论文里的其他病人（常常是一家人）也一样。

还有留出集（第 06 章）：它要等系统定稿才考一次。如果病例库里放了留出集的病人，开发时的每一次考试都用到了它们，留出集就不"干净"了。

### 用了什么技术，为什么
- **遮住本篇论文**：和其他工具一样（第 10 章的遮挡），考试时这道题出处论文里的病人全部不找
- **病例库只用 design 那一半**：开发和最后考留出集时都一样。留出集的病人一个都不进病例库
- 给 MCP 服务用的版本不是考试，用全部一万多个病人

### 公司里什么情况怎么做
- **检索类工具最容易泄露**：考题、或和考题同源的数据，进了被检索的库，分数就是假的。公司里会专门检查"测试集有没有混进检索库"
- **说清楚偏乐观的地方**：病例库和考题都是 Phenopacket Store 的人按同一套习惯标注的，相似度可能比真实世界高。医生自己写的症状，用词会更乱。这一点要写进结论里，不要藏

### 执行

In [ ]:
import collections
print("病例库里的病人：", collections.Counter(c.split for c in find_similar.bank))
with masking.masked([case.pmid]):
    found = find_similar(case.hpo_ids, top_k=50)
same_paper = sum(c.pmid == case.pmid for c in bank)
print(f"这道题的论文 {case.pmid} 在病例库里有 {same_paper} 个病人；遮住以后找到的 50 人里，来自它的：",
      sum(p.source == case.pmid for p in found.patients))

### 结果怎么看
- 病例库里只有 design 的病人，留出集一个都没有
- 这道题的论文有 9 个病人在库里，遮住以后一个都找不到

### 面试怎么说
> "Retrieval is where leakage hides: the bank contains the test cases themselves. Each case's own paper is masked, and the bank holds the design split only, so the heldout set stays unseen until the final run. I also report the optimism that remains: bank and queries share one curation style."

---

## 第 6 步 · 接进 agent 和 MCP，能拿掉

### 在干什么
把新工具交给 agent，也加进 MCP 服务；同时留一个开关，能把它拿掉。

### 背景
第 16 章定的考法是：全部工具做完以后，一起考一次，再每次拿掉一个考（消融实验）。所以每个新工具从一开始就要"能拿掉"：拿掉以后，提示词里也不能再提它，否则模型会去调一个不存在的工具。

### 用了什么技术，为什么
- **提示词里的工具说明按实际给的工具生成**：只给原来四个工具时，提示词和第 15、16 章一字不差（有测试检查），之前的成绩仍然可比
- **`raredx eval run --without find_similar_patients`**：拿掉这个工具，记录的名字和参数里都写明
- **MCP 服务**多了第五个工具（第 13 章），Claude Code 等应用也能直接用

### 执行

In [ ]:
from raredx.agent import system_prompt

core = ["rank_diseases", "check_match", "lookup_disease", "find_disease"]
full = system_prompt([*core, "find_similar_patients"])
print("\n".join(line for line in full.splitlines() if line.startswith("- ")))
print("\n拿掉以后，提示词里还提不提它：", "find_similar_patients" in system_prompt(core))

smoke = next(r for r in harness.runs("ci-20") if r["name"] == "seeded-agent"
             and "find_similar_patients" in harness.client.get_run(r["run_id"]).data.params["tools"])
traces = harness.traces(smoke["run_id"])
used = sum(any(s.name == "find_similar_patients" for s in t.data.spans) for t in traces)
m = harness.client.get_run(smoke["run_id"]).data.metrics
print(f"\nci-20 试跑：交了答案 {m['n_answered']:.0f}/20，其中 {used} 道调用了 find_similar_patients")

### 结果怎么看
- 提示词里多了一行，告诉 agent 这个工具什么时候用："和 rank_diseases 一起调，几个相似病人都得的病是很强的候选"
- 拿掉以后，提示词里不再提它
- ci-20 试跑：20 道全部交了答案，每道都调用了新工具。分数留到第 20 章大考再看

### 面试怎么说
> "Every new tool is built to be removable: the prompt's tool lines are generated from the tools given, so the ablation runs are clean, and with the original four tools the prompt is byte-identical to the one already measured."

---

## 本章小结

| 做了什么 | 结果 |
|---|---|
| 新工具 `find_similar_patients` | 拿病人和 6,014 个已发表的病人比，返回最像的人、他们的病、共同症状、出处 |
| 不用模型先考 | 第 1 名 29.0%（按病排名 19.0%），逐题比 35 比 15，`p = 0.007` |
| 两种方法合起来 | 前 10 名 59.5%（单独用都不到 50%） |
| 考试公平 | 遮住本篇论文；病例库只用 design，留出集不碰 |
| 接进 agent 和 MCP | 能拿掉；ci-20 试跑 20 道全部交卷 |

## 面试题

1. **Is this RAG?**
   > "Yes — retrieval before the model reasons — but over patients, not text. The data is structured as HPO terms, so similarity is computed on the ontology directly; no embeddings, no vector store to maintain."

2. **How do you keep a retrieval tool from leaking the answer?**
   > "The bank contains the test cases. Each case's source paper is masked, so neither the patient nor relatives reported with them can be found, and the bank holds the design split only, so the heldout set is never touched before the final run."

3. **What would make this number wrong in the real world?**
   > "Bank and queries come from the same curators, who code similar patients similarly; a clinician's free-text findings would match less neatly. That optimism is stated next to the result."

**下一章 18 · 疾病介绍（RAG）**：在疾病的文字介绍里搜，回答带出处。这是公司里最常见的那种 RAG：文档切块、向量检索，再单独评测检索准不准。